In [ ]:
pip install pandas numpy faiss-cpu sentence-transformers scikit-learn nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================
# IMPORTS
# =========================================
import pandas as pd
import numpy as np
import re
import string
import json
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# =========================================
# STEP 1: LOAD DATA
# =========================================
df = pd.read_csv("complaints.csv")
df.columns = df.columns.str.strip()

FileNotFoundError: [Errno 2] No such file or directory: 'complaints.csv'

In [ ]:
import zipfile
import os

# Define the path to your zip file and the extraction directory
zip_file_path = "/content/drive/MyDrive/complaints.csv.zip"  # Update this with the actual path
extraction_dir = "/content/unzipped_complaints"

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

print(f"'{zip_file_path}' unzipped to '{extraction_dir}' successfully.")
print("Contents of the unzipped directory:")
print(os.listdir(extraction_dir))

'/content/drive/MyDrive/complaints.csv.zip' unzipped to '/content/unzipped_complaints' successfully.
Contents of the unzipped directory:
['complaints.csv']


In [ ]:
import pandas as pd

chunk_size = 50000  # adjust (20k–100k based on RAM)

df = pd.read_csv("/content/unzipped_complaints/complaints.csv", chunksize=chunk_size)

In [ ]:
# Get the first chunk from the TextFileReader object
first_chunk = next(df)

# Now you can display the head of this chunk
display(first_chunk.head())

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,Yes,NaN,3477549
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
4,2024-01-21,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27401,Servicemember,Consent not provided,Web,2024-01-21,Closed with explanation,Yes,NaN,8191825


In [ ]:
# =========================================
# STEP 2: CLEAN DATA
# =========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
# Keep required columns
first_chunk = first_chunk[[
    "Consumer complaint narrative",
    "Company response to consumer"
]]

# Drop missing
first_chunk = first_chunk.dropna()
first_chunk = first_chunk[first_chunk["Consumer complaint narrative"].str.strip() != ""]

In [ ]:
# Apply the clean_text function to create 'clean_complaint' column
first_chunk["clean_complaint"] = first_chunk["Consumer complaint narrative"].apply(clean_text)

# Remove duplicates from the first_chunk
first_chunk = first_chunk.drop_duplicates(subset=["clean_complaint"]).reset_index(drop=True)

print("Cleaned data size:", first_chunk.shape)

Cleaned data size: (10789, 3)


In [ ]:
# =========================================
# STEP 3: LOAD EMBEDDING MODEL
# =========================================
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(first_chunk["clean_complaint"].tolist(), show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# =========================================
# STEP 4: FAISS INDEX
# =========================================
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
# =========================================
# STEP 5: CACHE (LOG FILE)
# =========================================
CACHE_FILE = "cache.json"

def load_cache():
    try:
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    except:
        return {}

def save_cache(cache):
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=4)

def normalize(text):
    return text.lower().strip()


In [ ]:
# =========================================
# STEP 6: SEMANTIC CACHE CHECK
# =========================================
def semantic_cache_check(query_embed, cache, threshold=0.9):
    if len(cache) == 0:
        return None

    cached_texts = list(cache.keys())
    cached_embeds = model.encode(cached_texts)

    sims = cosine_similarity([query_embed], cached_embeds)
    max_sim = sims[0].max()

    if max_sim > threshold:
        idx = sims[0].argmax()
        return cache[cached_texts[idx]]

    return None

In [ ]:
# =========================================
# STEP 7: RESPONSE FUNCTION
# =========================================
def get_response(user_input):
    cache = load_cache()
    norm_input = normalize(user_input)

    # 🔹 1. Exact Cache Check
    if norm_input in cache:
        print("⚡ Exact cache hit")
        return cache[norm_input]

    # 🔹 2. Semantic Cache Check
    query_embed = model.encode(user_input)
    cached_response = semantic_cache_check(query_embed, cache)

    if cached_response:
        print("⚡ Semantic cache hit")
        return cached_response

    # 🔹 3. FAISS Search
    D, I = index.search(np.array([query_embed]), k=3)
    responses = df.iloc[I[0]]["Company response to consumer"]

    # Pick best response
    final_response = responses.iloc[0]

    # 🔹 4. Store in Cache
    cache[norm_input] = final_response
    save_cache(cache)

    return final_response

In [ ]:
# =========================================
# 9. ISSUE DETECTION
# =========================================
def detect_issue(text):
    text = text.lower()

    if any(word in text for word in ["not mine", "fraud", "unauthorized", "identity theft"]):
        return "unauthorized or fraudulent account activity"

    elif any(word in text for word in ["charged twice", "double charge", "refund"]):
        return "duplicate or incorrect billing charges"

    elif "credit report" in text:
        return "incorrect information on your credit report"

    elif "payment" in text:
        return "payment processing issue"

    else:
        return "the issue reported"
# =========================================
# 10. RESPONSE GENERATION
# =========================================
def generate_response(label, complaint):
    issue = detect_issue(complaint)
    label = label.lower()

    if "explanation" in label:
        return f"""We have carefully reviewed your complaint regarding {issue}. Based on our investigation, we did not find sufficient evidence to modify the current records.

However, if you believe this information is incorrect, you may submit supporting documents such as identity proof or transaction details for further verification. Our team will reassess the issue upon receiving additional information."""

    elif "non-monetary relief" in label:
        return f"""We have investigated your complaint regarding {issue}. Necessary corrections have been made to your account to resolve the issue.

Please review your account details to confirm the update. If you notice any further discrepancies, feel free to contact us for additional assistance."""

    elif "monetary relief" in label:
        return f"""We have reviewed your complaint regarding {issue} and identified the problem. A refund or financial adjustment has been initiated.

The updated amount will reflect in your account within a few business days. We apologize for the inconvenience caused."""

    else:
        return f"""We have received your complaint regarding {issue}. Our team is currently investigating the matter.

We will update you with a resolution as soon as possible. Thank you for your patience."""
# =========================================
# 11. MAIN RESPONSE FUNCTION
# =========================================
def get_response(user_input):
    cache = load_cache()
    norm_input = normalize(user_input)

    # 1. Exact cache
    if norm_input in cache:
        print("⚡ Exact cache hit")
        return cache[norm_input]

    # 2. Semantic cache
    query_embed = model.encode(user_input)
    cached_response = semantic_cache_check(query_embed, cache)

    if cached_response:
        print("⚡ Semantic cache hit")
        return cached_response

    # 3. FAISS search
    D, I = index.search(np.array([query_embed]), k=3)
    # The error was here: df.iloc needs to be first_chunk.iloc
    label = first_chunk.iloc[I[0]]["Company response to consumer"].iloc[0]

    # 4. Generate human response
    final_response = generate_response(label, user_input)

    # 5. Store in cache
    cache[norm_input] = final_response
    save_cache(cache)

    return final_response

# =========================================
# 12. CHAT LOOP
# =========================================
if __name__ == "__main__":
    print("\n💬 Complaint Resolution Bot Ready!\n")

    while True:
        user_query = input("Enter complaint (or 'exit'): ")

        if user_query.lower() == "exit":
            print("Exiting...")
            break

        response = get_response(user_query)
        print("\n🤖 Bot Response:\n", response)


💬 Complaint Resolution Bot Ready!

Enter complaint (or 'exit'): “I was charged twice for the same transaction on my credit card. I request a refund for the duplicate charge as soon as possible.”

🤖 Bot Response:
 We have carefully reviewed your complaint regarding duplicate or incorrect billing charges. Based on our investigation, we did not find sufficient evidence to modify the current records.

However, if you believe this information is incorrect, you may submit supporting documents such as identity proof or transaction details for further verification. Our team will reassess the issue upon receiving additional information.
Enter complaint (or 'exit'): exit
Exiting...


In [ ]:
# ==============================
# INSTALL REQUIRED LIBRARIES
# ==============================
!pip install deep-translator langdetect --quiet

# ==============================
# IMPORTS
# ==============================
from deep_translator import GoogleTranslator
from langdetect import detect

# ==============================
# SAMPLE CHATBOT RESPONSE FUNCTION
# (Replace this with your actual chatbot pipeline)
# ==============================
def chatbot_response(text):

    text = text.lower()

    if "charged twice" in text or "duplicate payment" in text:
        return "Your duplicate payment issue has been identified and escalated."

    elif "refund" in text:
        return "Your refund request has been registered successfully."

    elif "login" in text:
        return "Please reset your password and try again."

    else:
        return "We have received your complaint and our support team will contact you shortly."


# ==============================
# MULTILINGUAL RESPONSE FUNCTION
# ==============================
def multilingual_chatbot(user_input):

    try:
        # Detect user language
        detected_lang = detect(user_input)

        print("Detected Language:", detected_lang)

        # Translate input to English
        translated_input = GoogleTranslator(
            source='auto',
            target='en'
        ).translate(user_input)

        print("Translated Input:", translated_input)

        # Generate chatbot response in English
        english_response = chatbot_response(translated_input)

        print("English Response:", english_response)

        # Translate response back to user's language
        if detected_lang != 'en':

            final_response = GoogleTranslator(
                source='en',
                target=detected_lang
            ).translate(english_response)

        else:
            final_response = english_response

        return final_response

    except Exception as e:
        return f"Error: {str(e)}"


# ==============================
# MAIN EXECUTION
# ==============================
print("🌍 Multilingual Complaint Chatbot Ready!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot stopped.")
        break

    response = multilingual_chatbot(user_input)

    print("Bot:", response)
    print()